# Rotten Fruit Detector — SageMaker Studio Training (SSD EfficientDet D1)

Trains **SSD EfficientDet D1** with the TF Object Detection API (`src/dataset.py` + `src/model.py` + `src/train.py`). `main()` builds the label map + TFRecords from the raw Roboflow CSV export, configures the pipeline, then trains, evaluates and exports.

Run from inside the cloned repo. Expected `dataset.zip` contents:

```text
train/_annotations.csv + *.jpg
valid/_annotations.csv + *.jpg
test/_annotations.csv  + *.jpg
```
The split folders may sit at the zip root or one level deep — the notebook auto-detects either.

**Cost tips:** smoke-test with `num_steps=200` first, then the real run. Setup (API install + weights) takes ~5 min of GPU time. Switch variants with `model_name="efficientdet_d0"` (cheaper) or `"efficientdet_d5"` (pricier). Pick a **GPU** kernel (e.g. multi-GPU `ml.g4dn`/`ml.g5`); training auto-uses all GPUs (MirroredStrategy), so scale `batch_size` with GPU count (below assumes 2). Stop the kernel when done to cap cost.

In [ ]:
# 1. Install system + python deps
%pip install -q -r ../requirements.txt

In [ ]:
# 2. Download the raw dataset from S3, unzip it, and locate the split folders
import zipfile
from pathlib import Path

import boto3

S3_BUCKET = "YOUR_BUCKET"
S3_KEY = "datasets/dataset.zip"
PROJECT_DIR = Path.cwd().parent

zip_path = PROJECT_DIR / "dataset.zip"
boto3.client("s3").download_file(S3_BUCKET, S3_KEY, str(zip_path))

EXTRACT_DIR = PROJECT_DIR / "dataset"
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(EXTRACT_DIR)

# The zip may hold train/valid/test at its root OR nested one folder deep,
# so find the directory that actually contains the split folders.
def find_dataset_root(base):
    base = Path(base)
    for path in [base, *(p for p in base.rglob("*") if p.is_dir())]:
        if (path / "train").is_dir():
            return path
    raise FileNotFoundError(f"No 'train' folder found under {base}")

DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Dataset root:", DATA_DIR)
print("Contains:", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

In [ ]:
# 3. Prepare TFRecords, configure EfficientDet D1, then train -> evaluate -> export
#    First run also clones the Object Detection API and downloads the weights.
#    batch_size = 8 x (number of GPUs). Smoke test first: num_steps=200.
import os
import sys

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))

os.environ["TF_USE_LEGACY_KERAS"] = "1"  # TFOD API needs Keras 2 (tf-keras)

from train import main

main(data_dir=DATA_DIR, model_name="efficientdet_d1", num_steps=4000, batch_size=16)

In [ ]:
# 4. Upload the exported model back to S3
import sagemaker

session = sagemaker.Session()
uri = session.upload_data(
    path=str(PROJECT_DIR / "exported-models" / "efficientdet_d1"),
    bucket=session.default_bucket(),
    key_prefix="rotten-fruit-detection/exported-model",
)
print("Uploaded to:", uri)